# 05 — Validation framework + first models

**Validation = leave-one-test-out (LOTO)** (`src/validation.py`): hide 1 test, train on the other 84, predict it;
repeat for all 85. Matches the real task (estimate daily rates *between* tests in Jan–Sep 2025).

**Main metric: MAPE** (mean absolute % error). R² looks high for everything because wells differ a lot
(GL-07 makes 6× more than GL-03) — even a per-well average gets R² ≈ 0.98 — so R² hides the real differences.

| Method | Idea |
|---|---|
| 0. well mean | average of that well's other tests (naive) |
| 1. time interpolation | straight line between neighbouring tests (naive, no sensors) |
| B. physics | `q_oil = J·(P_res(t) − Pwf)`, P_res linear in time per well; water from WC trend; gas = oil × mean GOR |
| A. LightGBM raw | ML on sensor averages + choke + days + well id |
| A. LightGBM + physics | same + physics features |

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

test_table = pd.read_parquet(load.PROCESSED / 'test_features.parquet')

experiments = {
    '0 well mean': vfm.well_mean,
    '1 time interpolation': vfm.time_interpolation,
    'B physics': vfm.physics,
    'A LightGBM raw features': vfm.make_ml_method(vfm.RAW_FEATURES),
    'A LightGBM + physics features': vfm.make_ml_method(vfm.PHYSICS_FEATURES),
}

results, predictions = [], {}
for name, method in experiments.items():
    predictions[name], _ = V.run_experiment(name, test_table, method, results)

V.save_results(results, load.ROOT / 'outputs' / 'ablation_results.csv')
pd.DataFrame(results).round(2)

## Physics model: error per well (MAPE %)

In [ ]:
V.score_per_well(test_table, predictions['B physics']).round(2)

## Predicted vs measured (LOTO) — points on the diagonal = perfect

In [ ]:
show = ['1 time interpolation', 'B physics', 'A LightGBM + physics features']
fig, axes = plt.subplots(len(show), 3, figsize=(14, 4 * len(show)))
for r, name in enumerate(show):
    for c, target in enumerate(V.TARGETS):
        ax = axes[r, c]
        true, pred = test_table[target], predictions[name][target]
        ax.scatter(true, pred, s=12)
        lim = [true.min() * 0.9, true.max() * 1.05]
        ax.plot(lim, lim, 'k--', lw=1)
        ax.set_title(f'{name}: {target}', fontsize=9)
        ax.set_xlabel('measured')
        ax.set_ylabel('predicted')
fig.tight_layout()
plt.show()